# Hockey Career Trajectory Predictor — Evaluation

Analysis of the walk-forward backtest outputs (1990-91 → 2023-24).
Run `python -m src.backtest` first to produce `data/output/*`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT = Path('../data/output')
vet_preds = pd.read_parquet(OUT / 'veteran_predictions.parquet')
vet_seasons = pd.read_csv(OUT / 'veteran_season_metrics.csv')
rook_preds = pd.read_parquet(OUT / 'rookie_predictions.parquet')
rook_classes = pd.read_csv(OUT / 'rookie_class_metrics.csv')
veterans = pd.read_parquet(OUT / 'veteran_features.parquet')
rookies = pd.read_parquet(OUT / 'rookie_features.parquet')
plt.rcParams['figure.figsize'] = (11, 4)

## 1. Veteran model: accuracy over time

Success metric (PRD): **MAE < 0.08 PPG** league-wide, and beating the 'last season's PPG' baseline.

In [ ]:
m = vet_seasons.dropna(subset=['mae'])
print(f"Seasons evaluated : {len(m)}")
print(f"Mean MAE (model)  : {m['mae'].mean():.4f}")
print(f"Mean MAE (baseline): {m['mae_baseline'].mean():.4f}")
print(f"Mean R2           : {m['r2'].mean():.3f}")
print(f"Directional acc.  : {m['directional_accuracy'].mean():.1%}")

fig, ax = plt.subplots()
ax.plot(m['season'], m['mae'], marker='o', ms=3, label='Model')
ax.plot(m['season'], m['mae_baseline'], marker='o', ms=3, label='Baseline (last-season PPG)')
for _, r in m[m['shortened'].fillna('') != ''].iterrows():
    ax.axvline(r['season'], ls=':', color='gray', alpha=.6)
    ax.annotate(r['shortened'], (r['season'], ax.get_ylim()[1]), rotation=90, fontsize=7, va='top')
ax.set_ylabel('MAE (PPG)'); ax.legend(); ax.set_title('Veteran MAE by season — dotted lines = shortened seasons')
plt.show()

In [ ]:
# Biggest over/under-performers across the whole backtest
ev = vet_preds[vet_preds['eval_ok']].dropna(subset=['ppg_next']).copy()
ev['err'] = ev['ppg_next'] - ev['pred_ppg']
from src.constants import season_label
ev['season'] = ev['pred_season'].map(season_label)
cols = ['skaterFullName', 'season', 'ppg_last1', 'pred_ppg', 'ppg_next', 'err']
display(ev.nlargest(10, 'err')[cols])
display(ev.nsmallest(10, 'err')[cols])

## 2. Rookie model: draft-class walk-forward

Success metric (PRD): **Spearman ρ > 0.55** between predicted and actual rookie PPG.

In [ ]:
print(f"Classes evaluated : {len(rook_classes)}")
print(f"Mean MAE          : {rook_classes['mae'].mean():.4f}")
print(f"Mean Spearman rho : {rook_classes['spearman'].mean():.3f}")

fig, ax = plt.subplots()
sc = ax.scatter(rook_preds['pred_ppg'], rook_preds['ppg_rookie'],
                c=rook_preds['draft_ovr'], cmap='RdYlGn_r', s=12, alpha=.7)
lim = max(rook_preds['pred_ppg'].max(), rook_preds['ppg_rookie'].max())
ax.plot([0, lim], [0, lim], ls='--', color='gray')
plt.colorbar(sc, label='Draft position (overall)')
ax.set_xlabel('Predicted rookie PPG'); ax.set_ylabel('Actual rookie PPG')
ax.set_title('Every draft steal (and bust) the model saw coming')
plt.show()

In [ ]:
# Steals: predicted well above draft-slot expectation AND delivered
rp = rook_preds.copy()
rp['draft_decile'] = pd.qcut(rp['draft_ovr'], 10, labels=False, duplicates='drop')
rp['expectation'] = rp.groupby('draft_decile')['ppg_rookie'].transform('mean')  # empirical draft-slot expectation
rp['steal_score'] = rp['pred_ppg'] - rp['expectation']
rp['delivered'] = rp['ppg_rookie'] - rp['expectation']
rp['season'] = rp['rookie_season'].map(season_label)
cols = ['skaterFullName', 'season', 'draft_ovr', 'pred_ppg', 'ppg_rookie', 'steal_score']
print('Top steals (model high, draft low, player delivered):')
display(rp[rp['delivered'] > 0].nlargest(15, 'steal_score')[cols])
print('Top busts (top-10 pick, lowest actual):')
display(rp[rp['draft_ovr'] <= 10].nsmallest(15, 'ppg_rookie')[cols])

## 3. Interpretability: SHAP on the final veteran model

PRD narrative hook: *the aging curve is real* — `age`/`age_sq` should rank among the most important features.

In [ ]:
import shap
from src.features import VETERAN_FEATURES
from src.model import train_model
from src.constants import season_window, FIRST_BACKTEST_SEASON, LAST_BACKTEST_SEASON

final_window = season_window(FIRST_BACKTEST_SEASON, LAST_BACKTEST_SEASON)[-6:-1]
train = veterans[veterans['next_season_actual'].isin(final_window)]
trained = train_model(train, VETERAN_FEATURES, 'ppg_next', order_col='next_season_actual')

sample = train[VETERAN_FEATURES].sample(min(2000, len(train)), random_state=42)
explainer = shap.TreeExplainer(trained.model)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, show=False)
plt.show()
shap.dependence_plot('age', shap_values, sample, show=False)
plt.show()

## Findings

See `WRITEUP.md` for the full narrative.